# Amazon Bedrock AgentCore Runtime에서 관찰성과 함께 Amazon Bedrock 모델을 사용하는 LlamaIndex Agents 호스팅

## 개요

이 튜토리얼에서는 기본 관찰성 기능이 포함된 Amazon Bedrock AgentCore Runtime을 사용하여 LlamaIndex 에이전트를 호스팅하는 방법을 알아봅니다. LlamaIndex 에이전트를 AgentCore Runtime에 배포하고 모니터링 및 분석을 위한 telemetry 데이터를 자동으로 수집하는 과정을 살펴봅니다.

### 튜토리얼 세부 정보

| 정보                | 세부 정보                                                                         |
|:--------------------|:---------------------------------------------------------------------------------|
| 튜토리얼 유형       | 대화형                                                                             |
| 에이전트 유형       | 단일                                                                               |
| Agentic Framework   | LlamaIndex                                                                       |
| LLM 모델            | Anthropic Claude Haiku                                                            |
| 튜토리얼 구성 요소  | 관찰성과 함께 AgentCore Runtime에서 에이전트 호스팅                               |
| 튜토리얼 분야       | 범분야                                                                             |
| 예제 난이도         | 쉬움                                                                               |
| 사용 SDK            | Amazon BedrockAgentCore Python SDK 및 boto3                                       |

### 튜토리얼 아키텍처

이 튜토리얼에서는 자동 관찰성 기능과 함께 LlamaIndex 에이전트를 AgentCore Runtime에 배포하는 방법을 설명합니다.

데모를 위해 Amazon Bedrock 모델과 산술 tool을 사용하는 LlamaIndex FunctionAgent를 활용합니다.

### 튜토리얼 주요 기능

* Amazon Bedrock AgentCore Runtime에서 LlamaIndex Agents 호스팅
* Amazon Bedrock 모델 사용
* 자동 관찰성 및 tracing
* 기본 제공 telemetry 수집

## 사전 요구 사항

이 튜토리얼을 실행하려면 다음 항목이 필요합니다.
* Python 3.10+
* AWS credentials
* Amazon Bedrock AgentCore SDK
* LlamaIndex
* Amazon Bedrock에서 Claude Haiku 모델을 사용할 수 있는 권한

### Terminal에서:

`cd 06-workshops/06-AgentCore-observability/01-Agentcore-runtime-hosted/LlamaIndex`

`python -m venv venv`

`source venv/bin/activate`

### Notebook에서:

venv를 kernel로 선택합니다.

In [ ]:
!pip install -q --force-reinstall -U -r requirements.txt

## LlamaIndex 에이전트 생성 및 로컬 실험

에이전트를 AgentCore Runtime에 배포하기 전에 실험을 위해 로컬에서 개발하고 실행해 보겠습니다.

프로덕션 agentic application에서는 에이전트 생성 과정과 호출 과정을 분리해야 합니다. AgentCore Runtime에서는 에이전트 호출 부분에 `@app.entrypoint` decorator를 적용하여 Runtime의 entrypoint로 사용합니다.

In [ ]:
%%writefile llamaindex_agent.py

import warnings
warnings.filterwarnings("ignore", message=".*validate_default.*", category=UserWarning)
import os
import json
import argparse
import boto3
from llama_index.llms.bedrock_converse import BedrockConverse
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.observability.otel import LlamaIndexOpenTelemetry


# LlamaIndex용 OpenTelemetry 계측 초기화
instrumentor = LlamaIndexOpenTelemetry()
# 수신 시작
instrumentor.start_registering()

def multiply(a: int, b: int) -> int:
    """Multiple two integers and returns the result integer"""
    return a * b

def add(a: int, b: int) -> int:
    """Add two integers and returns the result integer"""
    return a + b

def get_bedrock_model():
    model_id = "anthropic.claude-3-5-haiku-20241022-v1:0"
    region = boto3.Session().region_name
    
    bedrock_model = BedrockConverse(
        model=model_id,
        region_name=region,
    )
    return bedrock_model

# 모델 초기화
bedrock_model = get_bedrock_model()

# 산술 에이전트 생성
agent = FunctionAgent(
    tools=[add, multiply],
    llm=bedrock_model,
)

async def llamaindex_agent_bedrock(payload):
    """
    페이로드로 에이전트를 호출합니다.
    """
    user_input = payload.get("prompt")
    response = await agent.run(user_input)
    return str(response)

if __name__ == "__main__":
    import asyncio
    parser = argparse.ArgumentParser()
    parser.add_argument("payload", type=str)
    args = parser.parse_args()
    response = asyncio.run(llamaindex_agent_bedrock(json.loads(args.payload)))
    print(response)

#### 로컬 에이전트 호출

In [ ]:
!python llamaindex_agent.py '{"prompt": "What is (121 + 2) * 5?"}'

## AgentCore Runtime 배포를 위한 에이전트 준비

이제 에이전트를 AgentCore Runtime에 배포해 보겠습니다. 이를 위해 다음 작업이 필요합니다.
* `from bedrock_agentcore.runtime import BedrockAgentCoreApp`으로 Runtime App 가져오기
* 코드에서 `app = BedrockAgentCoreApp()`으로 App 초기화하기
* 호출 함수에 `@app.entrypoint` decorator 적용하기
* `app.run()`을 사용하여 AgentCoreRuntime이 에이전트 실행을 제어하도록 하기

### Amazon Bedrock 모델을 사용하는 LlamaIndex Agent
AgentCore Runtime 배포를 위해 LlamaIndex Agent를 준비해 보겠습니다.

In [ ]:
%%writefile llamaindex_agent.py

import warnings
warnings.filterwarnings("ignore", message=".*validate_default.*", category=UserWarning)
import os
import json
import boto3
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from llama_index.llms.bedrock_converse import BedrockConverse
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.observability.otel import LlamaIndexOpenTelemetry


app = BedrockAgentCoreApp()

# LlamaIndex용 OpenTelemetry 계측 초기화
instrumentor = LlamaIndexOpenTelemetry(debug=True)
# 수신 시작
instrumentor.start_registering()

def multiply(a: int, b: int) -> int:
    """Multiple two integers and returns the result integer"""
    return a * b

def add(a: int, b: int) -> int:
    """Add two integers and returns the result integer"""
    return a + b

def get_bedrock_model():
    model_id = "anthropic.claude-3-5-haiku-20241022-v1:0"
    region = boto3.Session().region_name
    
    bedrock_model = BedrockConverse(
        model=model_id,
        region_name=region,
    )
    return bedrock_model

# 모델 초기화
bedrock_model = get_bedrock_model()

# 산술 에이전트 생성
agent = FunctionAgent(
    tools=[add, multiply],
    llm=bedrock_model,
)

@app.entrypoint
async def llamaindex_agent_bedrock(payload):
    """
    페이로드로 에이전트를 호출합니다.
    """
    user_input = payload.get("prompt")
    print("User input:", user_input)
    response = await agent.run(user_input)
    return str(response)

if __name__ == "__main__":
    app.run()

## 내부에서는 어떤 작업이 이루어질까요?

`BedrockAgentCoreApp`을 사용하면 다음 작업이 자동으로 수행됩니다.

* 포트 8080에서 수신 대기하는 HTTP server 생성
* 에이전트 요청을 처리하는 필수 `/invocations` endpoint 구현
* health check를 위한 `/ping` endpoint 구현
* 적절한 content type 및 response format 처리
* AWS 표준에 따른 오류 처리 관리
* **관찰성 및 telemetry 수집 자동 활성화**

## AgentCore Runtime에 에이전트 배포

`CreateAgentRuntime` 작업은 container image, environment variable 및 암호화 설정을 지정할 수 있는 포괄적인 구성 옵션을 지원합니다. 또한 protocol 설정(HTTP, MCP)과 권한 부여 메커니즘을 구성하여 client와 에이전트 간의 통신 방식을 제어할 수 있습니다.

**참고:** 운영 환경에서는 코드를 container로 패키징하고 CI/CD pipeline 및 IaC를 사용하여 ECR에 push하는 것이 모범 사례입니다.

이 튜토리얼에서는 Amazon Bedrock AgentCore Python SDK를 사용하여 artifact를 간편하게 패키징하고 AgentCore Runtime에 배포합니다.

### AgentCore Runtime 배포 구성

먼저 starter toolkit을 사용하여 entrypoint, 방금 생성한 execution role 및 requirements file로 AgentCore Runtime 배포를 구성합니다. 실행 시 Amazon ECR repository를 자동 생성하도록 starter kit도 구성합니다.

구성 단계에서는 애플리케이션 코드를 기반으로 Dockerfile이 생성됩니다.

In [ ]:
from bedrock_agentcore_starter_toolkit.notebook.runtime.bedrock_agentcore import Runtime
from boto3.session import Session

boto_session = Session()
region = boto_session.region_name

agentcore_runtime = Runtime()
agent_name = "llamaindex_bedrock_getting_started10"
response = agentcore_runtime.configure(
    entrypoint="llamaindex_agent.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=agent_name,
)
response

### AgentCore Runtime으로 에이전트 실행

Dockerfile이 준비되었으므로 에이전트를 AgentCore Runtime으로 실행해 보겠습니다. 이 과정에서 Amazon ECR repository와 관찰성이 자동으로 활성화된 AgentCore Runtime이 생성됩니다. 특정 라이브러리의 불필요한 trace를 제외하려면 아래 environment variable에 라이브러리를 추가할 수 있습니다.

In [ ]:
launch_result = agentcore_runtime.launch(
    env_vars={
        # 최소 구성 - 지나치게 많은 정보를 생성하는 계측만 비활성화
        "OTEL_PYTHON_DISABLED_INSTRUMENTATIONS": (
            "jinja2,urllib3,requests,httpx,redis,aiohttp-client"
            # POST /invocations trace를 제거하려면 이 목록에 "starlette"를 추가합니다. 참고: session tracking이 비활성화됩니다.
        )
    }
)

### AgentCore Runtime 상태 확인
AgentCore Runtime을 배포했으므로 배포 상태를 확인해 보겠습니다.

In [ ]:
import time

status_response = agentcore_runtime.status()
status = status_response.endpoint["status"]
end_status = ["READY", "CREATE_FAILED", "DELETE_FAILED", "UPDATE_FAILED"]
while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint["status"]
    print(status)
status

## AgentCore Runtime의 Tracing 활성화

AWS console에서 Amazon Bedrock AgentCore로 이동합니다. Build and Deploy 아래의 Agent Runtime을 클릭하고 에이전트를 선택합니다.

에이전트 화면에서 Tracing 섹션이 나올 때까지 아래로 이동합니다. trace가 CloudWatch로 전송되도록 이 기능을 활성화합니다.

![enable_tracing.png](images/llamaindex_enable_tracing.png)

### AgentCore Runtime 호출

마지막으로 payload를 사용하여 AgentCore Runtime을 호출할 수 있습니다. 그러면 observability dashboard에서 확인할 수 있는 telemetry 데이터가 자동으로 생성됩니다.

In [ ]:
invoke_response = agentcore_runtime.invoke({"prompt": "What is (121 + 2) * 5?"})
invoke_response

### 호출 결과 처리

이제 호출 결과를 애플리케이션에서 사용할 수 있도록 처리할 수 있습니다.

In [ ]:
from IPython.display import Markdown, display
import json

response_text = invoke_response["response"][0]
display(Markdown(response_text))

### boto3로 AgentCore Runtime 호출

AgentCore Runtime이 생성되었으므로 어떤 AWS SDK로든 호출할 수 있습니다. 예를 들어 boto3의 `invoke_agent_runtime` method를 사용할 수 있습니다.

In [ ]:
import boto3

agent_arn = launch_result.agent_arn
agentcore_client = boto3.client("bedrock-agentcore", region_name=region)

boto3_response = agentcore_client.invoke_agent_runtime(
    agentRuntimeArn=agent_arn,
    qualifier="DEFAULT",
    payload=json.dumps({"prompt": "What is 15 * 8?"}),
)
if "text/event-stream" in boto3_response.get("contentType", ""):
    content = []
    for line in boto3_response["response"].iter_lines(chunk_size=1):
        if line:
            line = line.decode("utf-8")
            if line.startswith("data: "):
                line = line[6:]
                print(line)
                content.append(line)
    display(Markdown("\n".join(content)))
else:
    try:
        events = []
        for event in boto3_response.get("response", []):
            events.append(event)
    except Exception as e:
        events = [f"Error reading EventStream: {e}"]
    display(Markdown(json.loads(events[0].decode("utf-8"))))

## Observability Dashboard

### 자동 Telemetry 수집

LlamaIndex 에이전트가 AgentCore Runtime에서 실행되면 telemetry 데이터가 자동으로 수집되어 Amazon CloudWatch로 전송됩니다. 여기에는 다음 항목이 포함됩니다.

- **에이전트 실행 trace**: 에이전트 의사 결정 과정의 전체 workflow
- **LLM 호출**: 입력/출력 token을 포함한 Bedrock 모델 호출
- **Tool 사용**: 함수 호출 및 그 결과
- **성능 metric**: latency, token 사용량 및 오류율

### CloudWatch에서 Trace 확인

에이전트의 관찰성 데이터를 확인하려면 다음 단계를 수행합니다.

1. AWS CloudWatch console로 이동합니다.
2. **GenAI Observability** dashboard로 이동합니다.
3. 에이전트 Runtime을 선택하여 trace와 metric을 확인합니다.

### 주요 관찰성 기능

- **Session tracking**: 여러 상호작용의 연관 관계 파악
- **오류 모니터링**: 문제 식별 및 디버깅
- **성능 분석**: 에이전트 응답 시간 최적화

### Amazon CloudWatch의 AgentCore Observability

AgentCore Runtime에서 호스팅되는 에이전트의 관찰성을 활성화하는 단계를 정리하면 다음과 같습니다.

- Amazon CloudWatch에서 Transaction Search 활성화
- Bedrock AgentCore Runtime에 에이전트를 배포할 때 requirements.txt 파일에 `aws-opentelemetry-distro` 포함

## GenAI Observability dashboard의 Bedrock AgentCore 개요

관찰성이 활성화된 모든 에이전트를 확인하고 기간별로 데이터를 필터링할 수 있습니다. 몇 가지 예는 다음과 같습니다.

![genai-observability.png](images/llamaindex_dashboard_view.png)

기본 dashboard에서는 다음과 같이 모든 에이전트의 Runtime metric을 확인할 수 있습니다.

![runtime-all-agent-metrics.png](images/llamaindex_runtime_metrics_total_view.png)

방금 배포한 에이전트를 클릭하면 해당 에이전트의 Runtime metric dashboard로 이동하며, 사용자 지정 기간으로 데이터를 필터링할 수도 있습니다.

![runtime-metrics-per-agent.png](images/llamaindex_runtime_metrics_view.png)

Sessions View 탭에서는 이 에이전트와 연결된 모든 session을 확인할 수 있습니다.

![Agent-sessions-view.png](images/llamaindex_sessions_view.png)

Trace View 탭에서는 Runtime에서 실행되는 이 에이전트의 trace 및 span 정보를 살펴볼 수 있습니다.

![Agentcore-trace.png](images/llamaindex_traces_view.png)

GenAI Observability dashboard의 여러 기능을 살펴보며 trace에 대한 자세한 정보를 확인하세요.


## 정리(선택 사항)

이제 생성한 AgentCore Runtime을 정리하겠습니다.

In [ ]:
launch_result.ecr_uri, launch_result.agent_id, launch_result.ecr_uri.split("/")[1]

In [ ]:
import boto3

agentcore_control_client = boto3.client("bedrock-agentcore-control", region_name=region)
ecr_client = boto3.client("ecr", region_name=region)

runtime_delete_response = agentcore_control_client.delete_agent_runtime(
    agentRuntimeId=launch_result.agent_id,
)

response = ecr_client.delete_repository(repositoryName=launch_result.ecr_uri.split("/")[1], force=True)

# 축하합니다!

다음 작업을 성공적으로 완료했습니다.

- 산술 tool을 사용하는 LlamaIndex 에이전트 생성
- Amazon Bedrock AgentCore Runtime에 에이전트 배포
- 자동 관찰성 및 telemetry 수집 활성화
- 에이전트 호출 및 trace 데이터 생성

이제 에이전트가 완전한 관찰성 기능과 함께 실행됩니다. 성능을 모니터링하고 문제를 디버깅하며 agentic application을 최적화할 수 있습니다.